# Coastal Shield — Step 1: EDA

Load Argo (optional cache), CalCOFI, iNaturalist; summarize; align on 0.5° grid weekly; overlay nitrate + chlorophyll with bloom dates.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# project root: coastal_shield/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_loader import DataLoader, print_all_summaries

In [ ]:
loader = DataLoader()

df_calcofi = loader.load_calcofi()

# Argo: set load_argo_if_missing=False to skip network; or first run downloads to data/raw/argo_ca_2015_2025.parquet
try:
    df_argo = loader.load_argo()
except Exception as e:
    print("Argo fetch skipped or failed:", e)
    df_argo = pd.DataFrame()

df_inat = loader.load_inaturalist()

In [ ]:
print_all_summaries(df_argo, df_calcofi, df_inat)

In [ ]:
# Unified weekly grid + engineered features + bloom_label
unified = loader.align_datasets(
    df_calcofi=df_calcofi,
    df_inat=df_inat,
    df_argo=df_argo if len(df_argo) else None,
    load_argo_if_missing=not len(df_argo),
)

print("unified shape:", unified.shape)
unified.head(), unified.describe().T

In [ ]:
# Time series: nitrate + chlorophyll (CalCOFI weekly means), bloom event dates from iNaturalist
fig, ax1 = plt.subplots(figsize=(12, 5))

plot_df = unified.sort_values("week_start")
agg = plot_df.groupby("week_start", as_index=False).agg(
    nitrate=("nitrate", "mean"),
    chlorophyll=("chlorophyll", "mean"),
)

ax1.plot(agg["week_start"], agg["nitrate"], color="steelblue", label="Nitrate (μM, mean)")
ax1.set_xlabel("Week")
ax1.set_ylabel("Nitrate", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")

ax2 = ax1.twinx()
ax2.plot(agg["week_start"], agg["chlorophyll"], color="seagreen", alpha=0.85, label="Chlorophyll")
ax2.set_ylabel("Chlorophyll", color="seagreen")
ax2.tick_params(axis="y", labelcolor="seagreen")

if len(df_inat):
    for d in df_inat["date"]:
        ax1.axvline(d, color="crimson", alpha=0.15, linewidth=0.8)
    ax1.axvline(df_inat["date"].iloc[0], color="crimson", alpha=0.35, linewidth=1.0, label="Bloom obs. (iNat)")

ax1.set_title("CalCOFI weekly nitrate + chlorophyll (CA coast) with iNaturalist bloom dates")
fig.tight_layout()
plt.show()